# Imports

In [ ]:
set.seed(42)
doParallel::registerDoParallel(cores = parallel::detectCores())

library(ADAPTS)
library(ADAPTSdata)
library(ADAPTSdata2)
library(preprocessCore)
library(pheatmap)
library(mclust)
library(dplyr)
library(data.table)
library(gdata)
library(tidyr)
library(biomaRt)
library(ggplot2)
library(ggpubr)
library(rstatix)
library(stringr)
library(edgeR)
library(limma)

# Download reference dataset and read in

In [ ]:
#downloaded from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE182381
# https://www.nature.com/articles/s42003-023-04623-6?fromPaywallRec=false#data-availability
pub <- read.csv("GSE182381_reference_sample.txt", sep="\t")
head(pub)


In [ ]:
#Format to dataframe
collapsed_df <- pub %>%
  pivot_longer(-GeneSymbol, names_to = "celltype", values_to = "value") %>%
  mutate(prefix = str_sub(celltype, 1, 12)) %>%
  group_by(GeneSymbol, prefix) %>%
  summarise(value = mean(value, na.rm = TRUE), .groups = "drop") %>%
  pivot_wider(names_from = prefix, values_from = value)


head(collapsed_df)

In [ ]:
write.csv(collapsed_df, "GSE182381_reference_sample_average.csv")

In [ ]:
rnames <- collapsed_df$GeneSymbol
collapsed_df2 <- collapsed_df[,-1]
collapsed_df2 <- as.data.frame(collapsed_df2)
row.names(collapsed_df2) <- rnames
head(collapsed_df2)


In [ ]:
#Create DGE list
x <- DGEList(counts = collapsed_df2, lib.size = colSums(collapsed_df2),
norm.factors = rep(1,ncol(collapsed_df2)), remove.zeros = FALSE)
class(x)
x <- calcNormFactors(x, method = "TMM")
head(x$counts)

In [ ]:
y<- cpm(x, log=TRUE)
head(y)

# Load normalized gene expression values

In [ ]:
counts <- read.csv("Disc_TMM_voom_normalized_counts_20260105.csv", stringsAsFactors=FALSE)
head(counts)

In [ ]:
genes <- read.csv("gene_annot.csv", stringsAsFactors=FALSE)
head(genes)

In [ ]:
dim(genes)
dim(counts)
genes_counts <- merge(genes, counts, by.x="ENSEMBL", by.y="X", all=TRUE)
# Identify rows that are duplicated (either the first or subsequent occurrence)
duplicated_rows <- duplicated(genes_counts$SYMBOL) | duplicated(genes_counts$SYMBOL, fromLast = TRUE)

# Keep only rows where neither the original nor the "fromLast" duplicated check is TRUE
non_duplicated_df <- genes_counts[!duplicated_rows, ]

head(non_duplicated_df)
head(genes_counts)
row.names(non_duplicated_df) <- non_duplicated_df$SYMBOL
# Columns to remove
cols_to_remove <- c("ENSEMBL", "ENTREZID","SYMBOL")

# Remove columns
genes_counts2 <- non_duplicated_df[,!(names(non_duplicated_df) %in% cols_to_remove)]
head(genes_counts2)

# ADAPT deconvolution


In [ ]:
#ADAPT deconvolution
#not all marker genes in this dataset so removed
cellCounts_GSE2 <- estCellPercent.DCQ(y, genes_counts2, number_of_repeats=10)
head(cellCounts_GSE2)
write.csv(cellCounts_GSE2, "BulkRNAseq_Placenta_GSE_deconvolution_20260115_log2cpm.csv")

In [ ]:
#Read in metadata for plotting
meta <- read.csv("Disc_cohort_metadata_20251219.csv",stringsAsFactors=FALSE)
head(meta)
head(meta$Sample)

In [ ]:
cellCounts_GSE_t <- t(cellCounts_GSE2)
meta_decon3 <- merge(meta, cellCounts_GSE_t, by.x="SampleID", by.y="row.names")
write.csv(meta_decon3,"BulkRNAseq_meta_cell_type_deconvolution_20260115_GSE182381.csv")
head(meta_decon3)